# analysis_window_presets_v1

目标：验证固定分析窗口已经沉到 `src/analysis`，覆盖 YTD、近 1/3/6/12 个月、2020、2022、2023-2024 和用户自定义窗口，并能进入 AnalysisReport/dashboard 链路。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import (
    analyze_model_effects_by_regime_and_window,
    build_analysis_window_table,
    build_analysis_windows,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    write_analysis_report,
)
from src.core import DatasetSnapshot
from src.dashboard import write_model_matrix_dashboard_export

EXPERIMENT = "analysis_window_presets_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)


In [2]:
windows = build_analysis_windows(
    start="2020-01-01",
    end="2026-06-09",
    custom_windows={
        "covid_crash": ("2020-02-19", "2020-03-23"),
        "test": ("2026-05-01", "2026-06-09"),
    },
)
window_table = build_analysis_window_table(windows, source="research.analysis_window_presets_v1")
window_table.to_csv(CARD / "window_presets.csv", index=False)
window_table


,schema_version,window,start,end,source
0,analysis.window_presets.v1,ytd,2026-01-01,2026-06-09,research.analysis_window_presets_v1
1,analysis.window_presets.v1,trailing_1m,2026-05-09,2026-06-09,research.analysis_window_presets_v1
2,analysis.window_presets.v1,trailing_3m,2026-03-09,2026-06-09,research.analysis_window_presets_v1
3,analysis.window_presets.v1,trailing_6m,2025-12-09,2026-06-09,research.analysis_window_presets_v1
4,analysis.window_presets.v1,trailing_12m,2025-06-09,2026-06-09,research.analysis_window_presets_v1
5,analysis.window_presets.v1,year_2020,2020-01-01,2020-12-31,research.analysis_window_presets_v1
6,analysis.window_presets.v1,year_2022,2022-01-01,2022-12-31,research.analysis_window_presets_v1
7,analysis.window_presets.v1,period_2023_2024,2023-01-01,2024-12-31,research.analysis_window_presets_v1
8,analysis.window_presets.v1,covid_crash,2020-02-19,2020-03-23,research.analysis_window_presets_v1
9,analysis.window_presets.v1,test,2026-05-01,2026-06-09,research.analysis_window_presets_v1


In [3]:
dates = pd.date_range("2020-01-01", "2026-06-09", freq="B")
rows = []
for model in ["ridge_mse", "gcn_mse"]:
    nav = 1.0
    benchmark = 1.0
    for i, date in enumerate(dates):
        if pd.Timestamp("2020-02-19") <= date <= pd.Timestamp("2020-03-23"):
            regime = "stress"
            base_return = -0.006
        elif date.year == 2022:
            regime = "downtrend"
            base_return = -0.0012
        elif date.year in (2023, 2024):
            regime = "bull"
            base_return = 0.0014
        elif date >= pd.Timestamp("2026-05-01"):
            regime = "neutral"
            base_return = 0.0005
        else:
            regime = "neutral"
            base_return = 0.0004
        if model == "gcn_mse":
            model_return = base_return + (0.0009 if regime == "bull" else -0.0022 if regime == "stress" else 0.0001)
        else:
            model_return = base_return + (0.0002 if regime == "stress" else 0.0)
        benchmark_return = base_return * 0.65
        nav *= 1.0 + model_return
        benchmark *= 1.0 + benchmark_return
        rows.append({
            "date": date,
            "model": model,
            "regime": regime,
            "nav": nav,
            "benchmark_nav": benchmark,
            "position": 0.7 if model == "gcn_mse" else 0.45,
            "turnover": 0.03 if model == "gcn_mse" else 0.015,
        })
daily = pd.DataFrame(rows)
effect = analyze_model_effects_by_regime_and_window(daily, windows=windows, baseline_model="ridge_mse")
report = build_regime_window_analysis_report(
    artifact_id=EXPERIMENT,
    effect_analysis=effect,
    model_comparison=compare_models(daily),
    regime_metrics=compute_regime_metrics(daily),
    window_metrics=compute_window_metrics(daily, windows=windows),
    summary={"status": "analysis_window_presets_completed"},
)
dataset = DatasetSnapshot(
    name="synthetic_fixed_window_panel",
    universe=("SOXL", "SOXX"),
    start="2020-01-01",
    end="2026-06-09",
    calendar_path="data/qlib/soxl/calendars/day.txt",
    data_paths=("synthetic_fixed_window_panel",),
)
export_result = write_analysis_report(
    report,
    CARD,
    dataset=dataset,
    extra_paths={"window_presets": "window_presets.csv"},
)
dashboard_payload = write_model_matrix_dashboard_export(CARD, output_path=CARD / "dashboard_export.json", top_n=5)
summary = {
    "schema_version": "research.analysis_window_presets_v1",
    "status": "completed",
    "window_count": len(windows),
    "window_names": list(windows),
    "analysis_summary_window_names": effect.summary["window_names"],
    "analysis_overview_schema": dashboard_payload["analysis_overview"]["schema_version"],
    "top_weakness": dashboard_payload["analysis_overview"].get("top_weakness", {}),
    "manifest_window_presets": export_result["paths"].get("window_presets"),
}
(CARD / "window_preset_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/contrib/evaluate.py:85: RuntimeWarning: divide by zero encountered in scalar divide
  information_ratio = mean / std * np.sqrt(N)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/contrib/evaluate.py:85: RuntimeWarning: divide by zero encountered in scalar divide
  information_ratio = mean / std * np.sqrt(N)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/contrib/evaluate.py:8

{'schema_version': 'research.analysis_window_presets_v1',
 'status': 'completed',
 'window_count': 10,
 'window_names': ['ytd',
  'trailing_1m',
  'trailing_3m',
  'trailing_6m',
  'trailing_12m',
  'year_2020',
  'year_2022',
  'period_2023_2024',
  'covid_crash',
  'test'],
 'analysis_summary_window_names': ['ytd',
  'trailing_1m',
  'trailing_3m',
  'trailing_6m',
  'trailing_12m',
  'year_2020',
  'year_2022',
  'period_2023_2024',
  'covid_crash',
  'test'],
 'analysis_overview_schema': 'analysis.overview.v1',
 'top_weakness': {'baseline_model': 'ridge_mse',
  'decision_hint': 'block_or_rework',
  'max_drawdown_pct_delta': 4.9012706387474125,
  'model': 'gcn_mse',
  'priority_rank': 13,
  'priority_score': 30.377182,
  'regime': '',
  'relative_return_pct_delta': -3.2255852865452628,
  'review_note': 'gcn_mse 在 year_2020 弱于基准 ridge_mse：相对收益和稳定性同时落后，应先重做该 regime/window 的信号或仓位适配。',
  'review_signal': 'weakness',
  'segment': 'year_2020',
  'segment_type': 'window',
  'stability_scor

In [4]:
assert "ytd" in windows
assert "trailing_1m" in windows
assert "trailing_3m" in windows
assert "trailing_6m" in windows
assert "trailing_12m" in windows
assert "year_2020" in windows
assert "year_2022" in windows
assert "period_2023_2024" in windows
assert "covid_crash" in windows
assert "test" in windows
assert summary["manifest_window_presets"] == "window_presets.csv"
assert summary["analysis_summary_window_names"] == list(windows)
assert summary["analysis_overview_schema"] == "analysis.overview.v1"


## 中文结论

- 做了什么：新增并验证 `src/analysis` 固定窗口预设，统一生成 YTD、近 1/3/6/12 个月、2020、2022、2023-2024 和用户自定义窗口。
- 核心指标：本 notebook 生成 10 个窗口，`window_presets.csv` 记录 `analysis.window_presets.v1` 契约，`analysis_summary.json` 记录 `window_names/window_definitions`。
- regime 差异：同一模型在 bull、stress、downtrend 和自定义窗口中的表现会进入 `window_metrics.csv`、`window_regime_metrics.csv`、`segment_effect_matrix.csv` 和 `analysis_overview`。
- 是否晋升：晋升的是固定窗口构造和追溯能力，不晋升任何交易策略或模型。
- 风险：历史窗口会按当前数据可用范围裁剪；如果数据不覆盖 2020 或 2022，对应窗口会跳过或裁剪，不能伪造不可用区间。
- 下一步：生产 notebook 复用 `build_analysis_windows`，再把真实 qlib recorder 或模型矩阵输出接入这些固定窗口。